In [8]:
import sys
import os
import multiprocessing
import logging
from tqdm import tqdm
from ananke.models.collection import Collection
from ananke.configurations.collection import MergeConfiguration
from ananke.configurations.events import EventRedistributionMode
import time
import os
import sys
import multiprocessing
import logging
from tqdm import tqdm
from ananke.models.collection import Collection
from ananke.configurations.collection import MergeConfiguration
from ananke.schemas.event import RecordType

In [2]:
import pandas as pd

In [6]:
pd.read_hdf('/ptmp/arego/LargeTracks/16.h5',key='hits')

,time,string_id,module_id,pmt_id,record_id,type
0,1334.255478,3,4,3,0,2
0,1334.198279,3,4,5,0,2
0,1334.236611,3,4,7,0,2
0,1243.244978,3,5,1,0,2
0,1243.058155,3,5,2,0,2
...,...,...,...,...,...,...
0,1362.449670,13,12,2,30,2
0,1362.436244,13,12,4,30,2
0,1362.684513,13,12,8,30,2
0,1362.555383,13,12,12,30,2


In [11]:

configuration = MergeConfiguration.parse_obj(
    {
        'in_collections': [
            {
                'type': 'hdf5',
                'data_path': '/ptmp/arego/LargeTracks/16.h5',
                'read_only':'False',
            },
            {
                'type': 'hdf5',
                'data_path': '/ptmp/arego/LargeBio/7000s/16.h5',
                'read_only':'False',
            },
            {
                'type': 'hdf5',
                'data_path': '/ptmp/arego/LargeElectrical/7000s/16.h5',
                'read_only':'False',
            },
        ],
        'out_collection': {
                'type': 'hdf5',
                'data_path': 'MIX_16.h5',
                'read_only':'False',
        },
        'content': [
            {
                'primary_type': RecordType.REALISTIC_TRACK.value,
                'secondary_types': [RecordType.ELECTRICAL.value,RecordType.BIOLUMINESCENCE.value],
                'number_of_records': 20,
                'interval': {
                    'start': 0,
                    'end': 7000
                },
                
            },
        
           
        ]
    }
)

print(configuration)

in_collections=[HDF5StorageConfiguration(type=<StorageTypes.HDF5: 'hdf5'>, read_only=False, batch_size=100, data_path='/ptmp/arego/LargeTracks/16.h5', complevel=3, complib='lzo', optlevel=6), HDF5StorageConfiguration(type=<StorageTypes.HDF5: 'hdf5'>, read_only=False, batch_size=100, data_path='/ptmp/arego/LargeBio/7000s/16.h5', complevel=3, complib='lzo', optlevel=6), HDF5StorageConfiguration(type=<StorageTypes.HDF5: 'hdf5'>, read_only=False, batch_size=100, data_path='/ptmp/arego/LargeElectrical/7000s/16.h5', complevel=3, complib='lzo', optlevel=6)] tmp_collection=HDF5StorageConfiguration(type=<StorageTypes.HDF5: 'hdf5'>, read_only=False, batch_size=100, data_path='/raven/u/arego/olympus/lib/python3.10/site-packages/ananke/configurations/../../_tmp_efc1d696-0b24-48d9-877f-7c794fdfd5fbdata.h5', complevel=3, complib='lzo', optlevel=6) out_collection=HDF5StorageConfiguration(type=<StorageTypes.HDF5: 'hdf5'>, read_only=False, batch_size=100, data_path='MIX_16.h5', complevel=3, complib='lz

In [12]:
c=Collection.from_merge(configuration)

100it [04:26,  2.67s/it]              
100it [00:00, 103.45it/s]             
100it [00:00, 224.26it/s]             
100%|██████████| 20/20 [04:25<00:00, 13.30s/it]


In [16]:
with c:
    hits=c.storage.get_hits().df
hits

,time,string_id,module_id,pmt_id,record_id,type
0,620.621452,0,6,1,23,2
1,884.404619,0,6,1,23,2
2,427.451328,0,6,3,23,2
3,625.421052,0,6,3,23,2
4,888.274606,0,6,3,23,2
...,...,...,...,...,...,...
162002052,1821.675312,13,19,11,36,20
162002053,4685.256230,13,19,12,36,20
162002054,1777.977790,13,19,12,36,20
162002055,5357.815744,13,19,14,36,20


In [18]:
t=hits[hits['record_id']==25]['time']

In [19]:
t.max()-t.min()

6999.95273729487

In [20]:
hits.groupby("record_id")["time"].agg(lambda x: x.max() - x.min())

record_id
0     6999.875000
5     6999.764067
7     6999.750000
8     6999.290525
9     6999.936553
12    6999.750000
13    6999.875000
14    6999.687500
17    6999.875000
23    6999.764067
24    6999.402979
25    6999.952737
27    6999.482585
30    6999.303591
31    6999.733368
32    6999.875000
33    6999.733368
34    6999.303591
35    6999.482585
36    6999.500000
Name: time, dtype: float64